In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

dbutils.widgets.text("catalog_name", "workspace")
catalog = dbutils.widgets.get("catalog_name")

bronze_table = f"{catalog}.bronze.qualtrics_responses_raw"
silver_table = f"{catalog}.silver.qualtrics_responses"
quarantine_table = f"{catalog}.silver.qualtrics_responses_quarantine"

# 1. Read Bronze and standardize data types
bronze_df = spark.table(bronze_table)

typed_df = (
    bronze_df
    .withColumn("survey_id", F.trim(F.col("survey_id")))
    .withColumn("response_id", F.trim(F.col("response_id")))
    .withColumn("recorded_date", F.to_timestamp("recorded_date"))
    .withColumn("start_date", F.to_timestamp("start_date"))
    .withColumn("end_date", F.to_timestamp("end_date"))
    .withColumn("duration_seconds", F.col("duration_seconds").cast("int"))
    .withColumn("progress_pct", F.col("progress_pct").cast("int"))
    .withColumn("status", F.trim(F.col("status")))
    .withColumn("finished", F.col("finished").cast("boolean"))
    .withColumn("score", F.col("score").cast("double"))
    .withColumn("region", F.trim(F.col("region")))
    .withColumn("department", F.trim(F.col("department")))
    .withColumn("respondent_type", F.trim(F.col("respondent_type")))
    .withColumn("question_1", F.trim(F.col("question_1")))
    .withColumn("question_2", F.trim(F.col("question_2")))
    .withColumn("question_3", F.trim(F.col("question_3")))
    .withColumn("source_system", F.trim(F.col("source_system")))
    .withColumn("_silver_processed_ts", F.current_timestamp())
)

# 2. Evaluate production data quality rules
dq_df = typed_df.withColumn(
    "_dq_errors",
    F.array_compact(F.array(
        F.when(F.col("survey_id").isNull() | F.col("response_id").isNull(), F.lit("MISSING_PRIMARY_KEY")),
        F.when(F.col("end_date") < F.col("start_date"), F.lit("INVALID_DATE_RANGE")),
        F.when(F.col("duration_seconds") < 0, F.lit("NEGATIVE_DURATION")),
        F.when(~F.col("progress_pct").between(0, 100), F.lit("PROGRESS_OUT_OF_RANGE")),
        F.when(F.col("finished") & F.col("score").isNull(), F.lit("COMPLETED_MISSING_SCORE")),
        F.when((~F.col("finished")) & F.col("score").isNotNull(), F.lit("INCOMPLETE_HAS_SCORE")),
        F.when(F.col("_rescued_data").isNotNull(), F.lit("RESCUED_MALFORMED_DATA"))
    ))
)

# 3. Route failing rows to Quarantine (never drop silently)
quarantine_df = dq_df.filter(F.size("_dq_errors") > 0)
if quarantine_df.count() > 0:
    (
        quarantine_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(quarantine_table)
    )

# 4. Deduplicate valid records keeping latest recorded_date / _ingest_timestamp
window_spec = Window.partitionBy("survey_id", "response_id").orderBy(
    F.col("recorded_date").desc(),
    F.col("_ingest_timestamp").desc()
)

valid_deduped_df = (
    dq_df
    .filter(F.size("_dq_errors") == 0)
    .drop("_dq_errors", "_rescued_data")
    .withColumn("_rn", F.row_number().over(window_spec))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

# 5. Idempotent MERGE INTO Silver Delta table
if not spark.catalog.tableExists(silver_table):
    valid_deduped_df.write.format("delta").saveAsTable(silver_table)
else:
    target = DeltaTable.forName(spark, silver_table)
    (
        target.alias("t")
        .merge(
            valid_deduped_df.alias("s"),
            "t.survey_id = s.survey_id AND t.response_id = s.response_id"
        )
        .whenMatchedUpdateAll(condition="s.recorded_date >= t.recorded_date")
        .whenNotMatchedInsertAll()
        .execute()
    )

print(f"Silver valid rows: {spark.table(silver_table).count()} | Quarantined rows: {quarantine_df.count()}")

Silver valid rows: 120 | Quarantined rows: 0
